# Data Linkage and cleaning

---

First I have standardised the Account_id outputs with 'ACC-XXXXXXXX'

Then, I have removed all exactly identical rows - not rows that just share the same account number in order to avoid removing frequent loanees

Then I have combined all the columns into a large table, which combines all the values from all 3 tables

# Loading the datasets

In [3]:
from pathlib import Path
import pandas as pd


def find_data_directory(start: Path) -> Path:
    """Find the project data folder from the current notebook location."""
    required_files = {
        "applications.csv",
        "performance_12m.csv",
        "experian_consumer.csv",
    }

    for candidate in [start, *start.parents]:
        data_directory = candidate / "data"

        if data_directory.is_dir():
            available_files = {
                path.name for path in data_directory.iterdir()
            }

            if required_files.issubset(available_files):
                return data_directory

    raise FileNotFoundError(
        "Could not find a data folder containing "
        "applications.csv, performance_12m.csv and "
        "experian_consumer.csv."
    )


DATA_DIR = find_data_directory(Path.cwd().resolve())

applications = pd.read_csv(
    DATA_DIR / "applications.csv",
    dtype={"account_id": "string"},
)

performance = pd.read_csv(
    DATA_DIR / "performance_12m.csv",
    dtype={"account_id": "string"},
)

experian_consumer = pd.read_csv(
    DATA_DIR / "experian_consumer.csv",
    dtype={"account_id": "string"},
)


print(f"Data loaded from: {DATA_DIR}")
print(f"Applications: {applications.shape}")
print(f"Performance:  {performance.shape}")
print(f"Experian:     {experian_consumer.shape}")

Data loaded from: C:\Users\georg\OneDrive\Coding\Origination-Scorecard\data
Applications: (9987, 38)
Performance:  (10012, 13)
Experian:     (10000, 15)


---

# Cleaning and joining the data

In [5]:
from pathlib import Path
import pandas as pd


# Find the project root whether the notebook is opened from the root
# or from the notebooks folder.
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (
            (candidate / "data").is_dir()
            and (candidate / "pyproject.toml").is_file()
        ):
            return candidate

    raise FileNotFoundError("Could not find the project root")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "data"

ACCOUNT_ID_PATTERN = r"^ACC-\d{8}$"


# Load account IDs as text so their formatting is preserved.
applications = pd.read_csv(
    DATA_DIR / "applications.csv",
    dtype={"account_id": "string"},
)

performance = pd.read_csv(
    DATA_DIR / "performance_12m.csv",
    dtype={"account_id": "string"},
)

delphi_raw = pd.read_csv(
    DATA_DIR / "experian_consumer.csv",
    dtype={"account_id": "string"},
)


def make_exception_rows(
    dataframe,
    source_table,
    reason,
):
    """Copy excluded rows and record why they were removed."""

    exceptions = dataframe.loc[reason.notna()].copy()

    # Add 2 because pandas starts at zero and CSV row 1 is the header.
    exceptions.insert(
        0,
        "source_row_number",
        exceptions.index + 2,
    )

    exceptions.insert(
        0,
        "exception_reason",
        reason.loc[exceptions.index].astype(str),
    )

    exceptions.insert(
        0,
        "source_table",
        source_table,
    )

    return exceptions


def clean_source(
    dataframe,
    source_table,
    extra_invalid=None,
    extra_reason=None,
):
    """
    Remove rows containing:
    - Missing account IDs
    - Invalid account IDs
    - Any duplicated account ID

    duplicated(keep=False) removes every copy of a duplicated ID.
    """

    account_id = dataframe["account_id"]

    missing_id = (
        account_id.isna()
        | account_id.str.strip().eq("")
    )

    invalid_id = ~account_id.str.fullmatch(
        ACCOUNT_ID_PATTERN,
        na=False,
    )

    duplicate_id = (
        account_id.duplicated(keep=False)
        & ~missing_id
        & ~invalid_id
    )

    reason = pd.Series(
        pd.NA,
        index=dataframe.index,
        dtype="string",
    )

    reason.loc[missing_id] = "missing account_id"

    reason.loc[~missing_id & invalid_id] = (
        "invalid account_id format"
    )

    reason.loc[
        ~missing_id & ~invalid_id & duplicate_id
    ] = "duplicate account_id"

    # This allows source-specific validation, such as invalid
    # performance values or missing Delphi scores.
    if extra_invalid is not None:
        mask = reason.isna() & extra_invalid
        reason.loc[mask] = extra_reason

    exceptions = make_exception_rows(
        dataframe,
        source_table,
        reason,
    )

    clean_data = dataframe.loc[reason.isna()].copy()

    return clean_data, exceptions

## Data Linkage for the working set

This checks performance values, retains only the Delphi score, identifies unmatched records and performs exact matching.

In [6]:
# Identify the twelve monthly performance columns.
performance_columns = [
    column
    for column in performance.columns
    if column.startswith("missed_payment_m")
]

if not performance_columns:
    raise ValueError(
        "No monthly performance columns were found"
    )


# Monthly performance values must be either zero or one.
invalid_performance = ~performance[
    performance_columns
].isin([0, 1]).all(axis=1)


# Ignore every Delphi field except account ID and DelphiScore.
delphi = delphi_raw[
    ["account_id", "DelphiScore"]
].copy()

numeric_delphi_score = pd.to_numeric(
    delphi["DelphiScore"],
    errors="coerce",
)

invalid_delphi = numeric_delphi_score.isna()


# Apply the initial cleaning rules to each source.
applications_clean, application_exceptions = clean_source(
    applications,
    "applications",
)

performance_clean, performance_exceptions = clean_source(
    performance,
    "performance_12m",
    extra_invalid=invalid_performance,
    extra_reason="invalid monthly performance value",
)

delphi_clean, delphi_exceptions = clean_source(
    delphi,
    "experian_consumer",
    extra_invalid=invalid_delphi,
    extra_reason="missing or non-numeric DelphiScore",
)

delphi_clean["DelphiScore"] = pd.to_numeric(
    delphi_clean["DelphiScore"]
)


# Create sets of valid, unique account IDs.
application_ids = set(applications_clean["account_id"])
performance_ids = set(performance_clean["account_id"])
delphi_ids = set(delphi_clean["account_id"])


def unmatched_reason(account_ids, other_sources):
    """Explain which required sources do not contain an exact match."""

    reasons = []

    for account_id in account_ids:
        missing_from = [
            source_name
            for source_name, valid_ids in other_sources.items()
            if account_id not in valid_ids
        ]

        if missing_from:
            reasons.append(
                "no exact match in "
                + " and ".join(missing_from)
            )
        else:
            reasons.append(pd.NA)

    return pd.Series(
        reasons,
        index=account_ids.index,
        dtype="string",
    )


# Find valid-looking rows that cannot be exactly joined.
application_unmatched = unmatched_reason(
    applications_clean["account_id"],
    {
        "performance_12m": performance_ids,
        "experian_consumer": delphi_ids,
    },
)

performance_unmatched = unmatched_reason(
    performance_clean["account_id"],
    {
        "applications": application_ids,
        "experian_consumer": delphi_ids,
    },
)

delphi_unmatched = unmatched_reason(
    delphi_clean["account_id"],
    {
        "applications": application_ids,
        "performance_12m": performance_ids,
    },
)


# Add unmatched records to the appropriate exceptions tables.
application_exceptions = pd.concat(
    [
        application_exceptions,
        make_exception_rows(
            applications_clean,
            "applications",
            application_unmatched,
        ),
    ],
    ignore_index=True,
)

performance_exceptions = pd.concat(
    [
        performance_exceptions,
        make_exception_rows(
            performance_clean,
            "performance_12m",
            performance_unmatched,
        ),
    ],
    ignore_index=True,
)

delphi_exceptions = pd.concat(
    [
        delphi_exceptions,
        make_exception_rows(
            delphi_clean,
            "experian_consumer",
            delphi_unmatched,
        ),
    ],
    ignore_index=True,
)


# Only IDs found exactly once in all three clean sources are retained.
common_ids = (
    application_ids
    & performance_ids
    & delphi_ids
)

applications_for_join = applications_clean[
    applications_clean["account_id"].isin(common_ids)
].copy()

performance_for_join = performance_clean[
    performance_clean["account_id"].isin(common_ids)
].copy()

delphi_for_join = delphi_clean[
    delphi_clean["account_id"].isin(common_ids)
].copy()

## Geneate the new data tables for the joined data and the exceptions.

In [8]:
# Perform exact one-to-one joins.
working_data = (
    applications_for_join
    .merge(
        performance_for_join,
        on="account_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        delphi_for_join,
        on="account_id",
        how="inner",
        validate="one_to_one",
    )
    .sort_values("account_id")
    .reset_index(drop=True)
)


# Combine all excluded source rows into one auditable table.
exceptions = pd.concat(
    [
        application_exceptions,
        performance_exceptions,
        delphi_exceptions,
    ],
    ignore_index=True,
    sort=False,
)

exceptions = (
    exceptions
    .sort_values(
        ["source_table", "source_row_number"],
        kind="stable",
    )
    .reset_index(drop=True)
)


# Final validation checks.
assert working_data["account_id"].is_unique
assert working_data["account_id"].notna().all()

assert working_data["account_id"].str.fullmatch(
    ACCOUNT_ID_PATTERN
).all()

assert len(working_data) == len(common_ids)

# Confirm that no detailed Delphi fields entered working data.
ignored_delphi_columns = [
    column
    for column in working_data.columns
    if column.startswith("E1")
    or column.startswith("EA")
]

assert ignored_delphi_columns == []


# Reconcile every source row:
# each working account represents one row from each of three sources.
source_row_count = (
    len(applications)
    + len(performance)
    + len(delphi)
)

accounted_for_rows = (
    3 * len(working_data)
    + len(exceptions)
)

assert accounted_for_rows == source_row_count


# Write the requested output files.
working_data.to_csv(
    DATA_DIR / "working data.csv",
    index=False,
)

exceptions.to_csv(
    DATA_DIR / "exceptions.csv",
    index=False,
)


print(f"Working-data rows: {len(working_data):,}")
print(f"Exception rows: {len(exceptions):,}")

display(
    exceptions
    .groupby(["source_table", "exception_reason"])
    .size()
    .rename("rows")
)

Working-data rows: 9,888
Exception rows: 335


source_table       exception_reason                 
applications       duplicate account_id                 24
                   invalid account_id format            25
                   no exact match in performance_12m    50
experian_consumer  no exact match in applications       62
                   no exact match in performance_12m    50
performance_12m    invalid account_id format            37
                   missing account_id                   25
                   no exact match in applications       62
Name: rows, dtype: int64